In [4]:
import os
import io
import pandas as pd
import numpy as np
import math
import h5py
import sys
import cv2
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
import pickle
import lmdb
from collections import Counter
from torch.utils.data import Dataset

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
data_to_split = pd.read_csv('../metadata/train_val_metadata.csv')

In [6]:
number_of_folds = 5
folds = []

In [5]:
# findig the images for the patients
patient_images = pd.read_csv('../patient-id-image-data/csv/train_val_COVIDx_CT-3A.csv', sep=' ',header=0)
full_set = pd.DataFrame(columns=data_to_split.columns.values.tolist())
data_to_split['images'] = ' '
full_set['images'] = ' '

In [6]:
# matching images to specific patient IDs
for i, row in data_to_split.iterrows():
    mask = patient_images['fname'].str.match(pat='{}[^A-Za-z0-9]'.format(row['patient id']))
    images = np.array(patient_images[mask])
    if images.size == 0:
        continue
    else:
        full_set.loc[i] = row
        full_set.at[i,'images'] = images[:,0]

In [7]:
full_set.reset_index(drop=True, inplace=True)

In [8]:
image_count = 0
for i in range(0, len(full_set)):
    image_count = image_count + full_set['images'][i].size

In [9]:
# creating folds for training the model
already_sampled = pd.DataFrame(columns=full_set.columns.values.tolist())
split_image_count = 0
for i in range(0,number_of_folds):
    print("Now at fold: " + str(i))
    training_set = pd.DataFrame(columns=full_set.columns.values.tolist())
    val_set = pd.DataFrame(columns=full_set.columns.values.tolist())
    
    training_set = full_set

    j = 0
    if(i < 3):
        while(j < int(len(full_set)/number_of_folds)):
            sample = training_set.sample(n=1)
            if sample['patient id'].isin(already_sampled['patient id']).values[0]:
                continue
            else:
                val_set = pd.concat([val_set,sample])
                already_sampled = pd.concat([already_sampled,sample])
                training_set = training_set.drop(sample.index, inplace=False)
                split_image_count = split_image_count + sample.iloc[0]['images'].size
                j = j + 1
    else:
        while(j < math.ceil(len(full_set)/number_of_folds)):
            sample = training_set.sample(n=1)
            if sample['patient id'].isin(already_sampled['patient id']).values[0]:
                continue
            else:
                val_set = pd.concat([val_set,sample])
                already_sampled = pd.concat([already_sampled,sample])
                training_set = training_set.drop(sample.index, inplace=False)
                split_image_count = split_image_count + sample.iloc[0]['images'].size
                j = j + 1

    
    temp = []
    temp.append(training_set)
    temp.append(val_set)
    folds.append(temp)
    print("Already sampled: " + str(len(already_sampled)))
    print("Images added to splits: " + str(split_image_count))

Now at fold: 0
Already sampled: 902
Images added to splits: 74498
Now at fold: 1
Already sampled: 1804
Images added to splits: 155543
Now at fold: 2
Already sampled: 2706
Images added to splits: 230450
Now at fold: 3
Already sampled: 3609
Images added to splits: 305933
Now at fold: 4
Already sampled: 4512
Images added to splits: 381596


In [10]:
# Checking if everything is sampled
split_image_count == image_count

True

In [ ]:
# label distribution in TRAIN and VAL sets

In [ ]:
# Distribution of class labels in training data
for i in range(0, len(folds)):
    print(folds[i][0]['finding'].value_counts(normalize=True))

In [ ]:
# Distribution of class labels in validation data
for i in range(0, len(folds)):
    print(folds[i][1]['finding'].value_counts(normalize=True))

In [11]:
# creating the TEST fold

In [12]:
test_metadata = pd.read_csv('../metadata/test_metadata.csv')

In [13]:
# findig the images for the patients in the TEST fold
test_patient_images = pd.read_csv('../patient-id-image-data/csv/test_COVIDx_CT-3A.csv', sep=' ',header=0)
test_set = pd.DataFrame(columns=test_metadata.columns.values.tolist())
test_metadata['images'] = ' '
test_set['images'] = ' '

In [14]:
# matching images to specific patient IDs in the TEST set
for i, row in test_metadata.iterrows():
    mask = test_patient_images['fname'].str.match(pat='{}[^A-Za-z0-9]'.format(row['patient id']))
    images = np.array(test_patient_images[mask])
    if images.size == 0:
        continue
    else:
        test_set.loc[i] = row
        test_set.at[i,'images'] = images[:,0]

In [15]:
test_set

,patient id,source,country,sex,age,finding,verified finding,slice selection,view,modality,images
0,CP_0,CNCB,China,NaN,NaN,Pneumonia,Yes,Expert,Axial,CT,CP_0_3136_0207.png
1,CP_1070,CNCB,China,NaN,NaN,Pneumonia,Yes,Expert,Axial,CT,"[CP_1070_3112_0032.png, CP_1070_3112_0033.png,..."
2,CP_1091,CNCB,China,NaN,NaN,Pneumonia,Yes,Expert,Axial,CT,"[CP_1091_3309_0157.png, CP_1091_3309_0158.png,..."
3,CP_1093,CNCB,China,NaN,NaN,Pneumonia,Yes,Expert,Axial,CT,"[CP_1093_3311_0101.png, CP_1093_3311_0105.png,..."
4,CP_1100,CNCB,China,NaN,NaN,Pneumonia,Yes,Expert,Axial,CT,"[CP_1100_3318_0044.png, CP_1100_3318_0047.png,..."
...,...,...,...,...,...,...,...,...,...,...,...
498,COVIDCTMD-normal063,COVID-CT-MD,Iran,F,59.0,Normal,Yes,NaN,Axial,CT,"[COVIDCTMD-normal063-IM0001.png, COVIDCTMD-nor..."
499,COVIDCTMD-normal064,COVID-CT-MD,Iran,M,41.0,Normal,Yes,NaN,Axial,CT,"[COVIDCTMD-normal064-IM0001.png, COVIDCTMD-nor..."
500,COVIDCTMD-normal067,COVID-CT-MD,Iran,M,66.0,Normal,Yes,NaN,Axial,CT,"[COVIDCTMD-normal067-IM0001.png, COVIDCTMD-nor..."
501,COVIDCTMD-normal068,COVID-CT-MD,Iran,F,64.0,Normal,Yes,NaN,Axial,CT,"[COVIDCTMD-normal068-IM0001.png, COVIDCTMD-nor..."


In [16]:
# -----------------------------------------
# writing folds to disk for later reference
# -----------------------------------------

In [18]:
# open .txt file and write TRAINING data to txt file
for i in range(0,len(folds)):
    with open('../folds-txt/train_' + str(i) + '.txt', 'w+') as f:
        for item in folds[i][0].iterrows():
            f.write('%s\n' %item[1]['patient id'])
            f.write('%s\n' %item[1]['finding'])
            f.write('%s\n' %item[1]['images'])
# close the file
f.close()

In [19]:
# open .txt file and write VALIDATION data to txt file
for i in range(0,len(folds)):
    with open('../folds-txt/val_' + str(i) + '.txt', 'w+') as f:
        for item in folds[i][1].iterrows():
            f.write('%s\n' %item[1]['patient id'])
            f.write('%s\n' %item[1]['finding'])
            f.write('%s\n' %item[1]['images'])
# close the file
f.close()

In [ ]:
# ---------------------------
# writing data as data frames
# ---------------------------

In [20]:
# write TRAINING data to csv file
for i in range(0,len(folds)):
    folds[i][0].to_csv('../folds-csv/train_' + str(i) + '.csv')

In [21]:
# write VALIDATION data to csv file
for i in range(0,len(folds)):
    folds[i][1].to_csv('../folds-csv/val_' + str(i) + '.csv')

In [ ]:
# ------------------------
# writing test data to txt
# ------------------------

In [22]:
with open('../folds-txt/test.txt', 'w+') as f:
    for item in test_set.iterrows():
        f.write('%s\n' %item[1]['patient id'])
        f.write('%s\n' %item[1]['finding'])
        f.write('%s\n' %item[1]['images'])
# close the file
f.close()

In [23]:
# ------------------------
# writing test data to csv
# ------------------------

In [24]:
test_set.to_csv('../folds-csv/test.csv')

In [27]:
# Sanity check

In [26]:
for i in range(5):
    print(f"\nFold {i} TRAIN:")
    print(folds[i][0]["finding"].value_counts())

    print("===================================")

    print(f"Fold {i} VAL:")
    print(folds[i][1]["finding"].value_counts())

    print("===================================")
    print("===================================")


Fold 0 TRAIN:
finding
COVID-19     2765
Pneumonia     627
Normal        218
Name: count, dtype: int64
Fold 0 VAL:
finding
COVID-19     684
Pneumonia    167
Normal        51
Name: count, dtype: int64

Fold 1 TRAIN:
finding
COVID-19     2759
Pneumonia     648
Normal        203
Name: count, dtype: int64
Fold 1 VAL:
finding
COVID-19     690
Pneumonia    146
Normal        66
Name: count, dtype: int64

Fold 2 TRAIN:
finding
COVID-19     2745
Pneumonia     641
Normal        224
Name: count, dtype: int64
Fold 2 VAL:
finding
COVID-19     704
Pneumonia    153
Normal        45
Name: count, dtype: int64

Fold 3 TRAIN:
finding
COVID-19     2771
Pneumonia     623
Normal        215
Name: count, dtype: int64
Fold 3 VAL:
finding
COVID-19     678
Pneumonia    171
Normal        54
Name: count, dtype: int64

Fold 4 TRAIN:
finding
COVID-19     2756
Pneumonia     637
Normal        216
Name: count, dtype: int64
Fold 4 VAL:
finding
COVID-19     693
Pneumonia    157
Normal        53
Name: count, dtype: int64


In [27]:
# Saving to LMDB

In [28]:
CLASS_TO_IDX = {
    "Normal": 0,
    "Pneumonia": 1,
    "COVID-19": 2
}

In [29]:
def normalize_images(image_names):
    if isinstance(image_names, np.ndarray):
        if image_names.ndim == 0:
            return [image_names.item()]
        return image_names.tolist()
    if isinstance(image_names, str):
        return [image_names]
    return list(image_names)

In [31]:
def write_lmdb(df, lmdb_path, image_root, initial_map_size=2 * 1024**3):
    env = lmdb.open(
        lmdb_path,
        map_size=initial_map_size,
        subdir=False,
        meminit=False,
        map_async=True
    )

    idx = 0
    map_size = initial_map_size

    txn = env.begin(write=True)

    try:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            images = normalize_images(row["images"])
            label = CLASS_TO_IDX[row["finding"]]

            for img_name in images:
                img_path = os.path.join(image_root, img_name)

                if not os.path.isfile(img_path):
                    continue

                with Image.open(img_path) as img:
                    img = img.convert("RGB")
                    img = img.resize((224, 224), Image.BILINEAR)
                    buffer = io.BytesIO()
                    img.save(buffer, format="JPEG", quality=90)

                key = f"{idx}".encode()
                value = pickle.dumps({
                    "image": buffer.getvalue(),
                    "label": label
                })

                try:
                    txn.put(key, value)
                    idx += 1

                except lmdb.MapFullError:
                    txn.abort()
                    map_size *= 2
                    env.set_mapsize(map_size)
                    txn = env.begin(write=True)
                    txn.put(key, value)
                    idx += 1

        txn.put(b"__len__", pickle.dumps(idx))
        txn.commit()

    finally:
        env.sync()
        env.close()

In [32]:
IMAGE_ROOT = "../3A_images/"
OUTPUT_ROOT = "../lmdbs/"

In [33]:
# writing TRAIN and VAL images to LMDB
for i in range(number_of_folds):

    train_lmdb = os.path.join(
        OUTPUT_ROOT, f"fold_{i}_train.lmdb"
    )

    val_lmdb = os.path.join(
        OUTPUT_ROOT, f"fold_{i}_val.lmdb"
    )
    
    print(f"Writing fold {i} train LMDB")
    write_lmdb(folds[i][0], train_lmdb, IMAGE_ROOT)

    print(f"Writing fold {i} val LMDB")
    write_lmdb(folds[i][1], val_lmdb, IMAGE_ROOT)

Writing fold 0 train LMDB


100%|██████████████████████████████████████████████████████████████████████████████| 3610/3610 [34:38<00:00,  1.74it/s]


Writing fold 0 val LMDB


100%|████████████████████████████████████████████████████████████████████████████████| 902/902 [09:01<00:00,  1.67it/s]


Writing fold 1 train LMDB


...

100%|████████████████████████████████████████████████████████████████████████████████| 903/903 [07:19<00:00,  2.06it/s]


Writing fold 4 train LMDB


100%|██████████████████████████████████████████████████████████████████████████████| 3609/3609 [30:22<00:00,  1.98it/s]


Writing fold 4 val LMDB


100%|████████████████████████████████████████████████████████████████████████████████| 903/903 [08:44<00:00,  1.72it/s]


In [34]:
test_lmdb = os.path.join(OUTPUT_ROOT, "test.lmdb")

print("Writing test LMDB")
write_lmdb(test_set, test_lmdb, IMAGE_ROOT)

Writing test LMDB


100%|████████████████████████████████████████████████████████████████████████████████| 423/423 [03:02<00:00,  2.31it/s]


In [1]:
# SANITY CHECK FOR CLASS LABLES inside the LMDB datasets

In [5]:
class LMDBDataset(Dataset):
    def __init__(self, lmdb_path, transform=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Open once ONLY to read keys, then close
        env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            self.keys = [k for k, _ in txn.cursor() if k != b"__len__"]

        env.close()

        # Critical: length derived from keys, not __len__
        self.length = len(self.keys)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError

        # Open LMDB locally (safe for Windows)
        env = lmdb.open(
            self.lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            data = pickle.loads(txn.get(self.keys[idx]))

        env.close()

        img = Image.open(io.BytesIO(data["image"])).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, data["label"]

In [15]:
def label_distribution(lmdb_path):
    ds = LMDBDataset(lmdb_path)
    labels = [ds[i][1] for i in range(len(ds))]
    return Counter(labels)

for i in range(0, number_of_folds):
    print('===== FOLD: ' + str(i))
    print("TRAIN:", label_distribution('../lmdbs/fold_' + str(i) +'_train.lmdb'))
    print("VAL:",   label_distribution('../lmdbs/fold_' + str(i) + '_val.lmdb'))

===== FOLD: 0
TRAIN: Counter({2: 159417, 0: 4969, 1: 1875})
VAL: Counter({2: 59227, 0: 9143, 1: 6128})
===== FOLD: 1
TRAIN: Counter({2: 153849, 0: 4524, 1: 1648})
VAL: Counter({2: 62692, 0: 12628, 1: 5725})
===== FOLD: 2
TRAIN: Counter({2: 159892, 0: 4694, 1: 1541})
VAL: Counter({2: 58920, 0: 8174, 1: 7813})
===== FOLD: 3
TRAIN: Counter({2: 159018, 0: 4936, 1: 1492})
VAL: Counter({2: 59433, 1: 8882, 0: 7168})
===== FOLD: 4
TRAIN: Counter({2: 159720, 0: 3789, 1: 1772})
VAL: Counter({2: 60602, 0: 8631, 1: 6430})


In [14]:
 print("TEST:", label_distribution('../lmdbs/test.lmdb'))

TEST: Counter({0: 15968, 1: 7965, 2: 7437})
